# Results Summary

All metrics computed from cached aggregated statistics (derived from `predictions.pt` files).
Covers: overall MAE ranking, lead-time behaviour, masking effect, spatial attention,
NLL vs Huber σₑ, nearest-neighbour distance and topographic correlations.

In [ ]:
import os, sys, datetime as dt
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.spatial import distance_matrix

for _cand in (os.getcwd(),
              os.path.join(os.getcwd(), "notebooks", "analysis"),
              os.path.dirname(os.path.abspath("__file__"))):
    if os.path.isfile(os.path.join(_cand, "common.py")):
        if _cand not in sys.path: sys.path.insert(0, _cand)
        break
import importlib, common as C; importlib.reload(C)
plt.style.use("default")
plt.rcParams.update({"figure.dpi": 110, "font.size": 10,
                     "figure.facecolor": "white", "axes.facecolor": "white",
                     "savefig.facecolor": "white", "axes.edgecolor": "0.3",
                     "axes.labelcolor": "black", "xtick.color": "0.3",
                     "ytick.color": "0.3", "text.color": "black"})

CACHE = os.path.join(os.path.dirname(os.path.abspath("__file__")),
                     "..", "..", "analysis_outputs", "cache")
if not os.path.isdir(CACHE):
    CACHE = os.path.join(os.getcwd(), "analysis_outputs", "cache")

ns = C.norm_stats(); VARS = ns["var_names"]; STD = ns["std"]
stn = C.station_table(); KEEP = C.keep_mask(stn, VARS)
NV = len(VARS)

ORDER = ["lstm-baseline-v1", "v32-blind", "v31", "v27", "v30-nll"]
LABELS = [C.MODELS[r][0] for r in ORDER]
COLORS = [C.MODELS[r][1] for r in ORDER]

# Load MR=0.0 aggregates
AGG0 = {}
for r in ORDER:
    f = os.path.join(CACHE, f"agg_{r}_mr0.00.npz")
    AGG0[r] = dict(np.load(f))

# Load MR=0.5 aggregates
AGG5 = {}
for r in ["v27", "v30-nll"]:
    f = os.path.join(CACHE, f"agg_{r}_mr0.50.npz")
    AGG5[r] = dict(np.load(f))

# Load extended aggregates (TOD breakdowns)
EXT0 = {}
for r in ORDER:
    f = os.path.join(CACHE, f"agg_ext_{r}_mr0.00.npz")
    if os.path.isfile(f):
        EXT0[r] = dict(np.load(f))

GRID = AGG0["v27"]["grid"]  # lead-time grid in steps
K = len(GRID)
LEAD = C.lead_labels(GRID)
REF_KI = 6  # +3h
print(f"Lead grid ({K} points): {GRID}")
print(f"Models loaded: {list(AGG0.keys())}")
print(f"MR=0.5 models: {list(AGG5.keys())}")
print(f"Extended agg models: {list(EXT0.keys())}")

## 1. Overall MAE by model at MR=0.0

Network-pooled MAE in physical units, split into Δ=0 (reconstruction) and Δ>0 (forecasting).

In [ ]:
rows = []
for r in ORDER:
    a = AGG0[r]; label = C.MODELS[r][0]
    for vi, v in enumerate(VARS):
        cnt = a["mod_all_cnt"][:, :, vi]
        s   = a["mod_all_sum_phys"][:, :, vi]
        m0 = cnt[0] > 0
        mae0 = np.sum(s[0, m0]) / np.sum(cnt[0, m0])
        mfc = cnt[1:] > 0
        mae_fc = np.sum(s[1:][mfc]) / np.sum(cnt[1:][mfc])
        mall = cnt > 0
        mae_all = np.sum(s[mall]) / np.sum(cnt[mall])
        rows.append(dict(model=label, variable=v, unit=C.UNITS[v],
                         mae_d0=mae0, mae_fc=mae_fc, mae_all=mae_all))

df1 = pd.DataFrame(rows)
for c in ["mae_d0", "mae_fc", "mae_all"]:
    df1[c] = df1[c].map(lambda x: f"{x:.4f}")

print("Overall MAE at MR=0.0 (physical units)")
print("=" * 80)
for label in LABELS:
    sub = df1[df1.model == label]
    print(f"\n  {label}:")
    print(sub[["variable", "unit", "mae_d0", "mae_fc", "mae_all"]].to_string(index=False))

## 2. MAE vs lead time (MR=0.0)

Line plots per variable showing all models.

In [ ]:
leads_min = GRID * 10 / 60  # convert steps to hours

fig, axes = plt.subplots(1, NV, figsize=(4.5 * NV, 3.5), sharey=False)
for vi, (ax, v) in enumerate(zip(axes, VARS)):
    for ri, r in enumerate(ORDER):
        a = AGG0[r]
        mae_by_k = []
        for ki in range(K):
            cnt = a["mod_all_cnt"][ki, :, vi]
            s   = a["mod_all_sum_phys"][ki, :, vi]
            m = cnt > 0
            mae_by_k.append(np.sum(s[m]) / np.sum(cnt[m]))
        ax.plot(leads_min, mae_by_k, color=COLORS[ri], label=LABELS[ri], linewidth=1.5)
    ax.set_title(f"{v} [{C.UNITS[v]}]", fontsize=10)
    ax.set_xlabel("Lead time (h)")
    if vi == 0:
        ax.set_ylabel("MAE (physical)")
    ax.grid(alpha=0.3)

axes[-1].legend(fontsize=8, loc="upper left")
fig.suptitle("MAE vs lead time, all 155 stations visible (MR=0), pooled over 11,684 test windows (2023–2024)\n"
             "Δ=0 is reconstruction, Δ>0 forecast; physical units", fontsize=12, y=1.04)
plt.tight_layout()
C.save_fig(fig, "summary_mae_vs_lead")
plt.show(); plt.close(fig)

In [ ]:
KEY_LEADS = [(0, "Δ=0"), (1, "+30m"), (6, "+3h"), (12, "+6h")]
for vi, v in enumerate(VARS):
    print(f"\n{v} [{C.UNITS[v]}]:")
    header = f"  {'Model':25s}" + "".join(f"  {lbl:>8s}" for _, lbl in KEY_LEADS)
    print(header)
    for ri, r in enumerate(ORDER):
        a = AGG0[r]; label = LABELS[ri]
        vals = []
        for ki, _ in KEY_LEADS:
            cnt = a["mod_all_cnt"][ki, :, vi]
            s   = a["mod_all_sum_phys"][ki, :, vi]
            m = cnt > 0
            vals.append(np.sum(s[m]) / np.sum(cnt[m]))
        print(f"  {label:25s}" + "".join(f"  {v:8.4f}" for v in vals))

## 3. Effect of masking

### 3a. MR=0.5 masked vs visible gap (v27)

How much worse are masked stations compared to visible ones?  
The gap is huge at Δ=0 (pure reconstruction) and shrinks with lead time.

### 3b. Dense vs MAE at MR=0.0

The controlled comparison: both share the same architecture, only training
mask ratio differs (0.0 vs 0.5). Positive Δ = MAE worse than Dense.

In [ ]:
# ── 3a: Masked vs visible degradation at MR=0.5 ──
print("v27 at MR=0.5: masked vs visible MAE")
print("=" * 75)
KEY_LEADS_3 = [(0, "Δ=0"), (1, "+30m"), (6, "+3h"), (12, "+6h")]
a5 = AGG5["v27"]

for vi, v in enumerate(VARS):
    print(f"\n  {v} [{C.UNITS[v]}]:")
    print(f"  {'Lead':>8s}  {'visible':>10s}  {'masked':>10s}  {'gap':>10s}  {'gap %':>8s}")
    for ki, lbl in KEY_LEADS_3:
        cv = a5["mod_vis_cnt"][ki, :, vi]; sv = a5["mod_vis_sum_phys"][ki, :, vi]
        mv = cv > 0; mae_v = np.sum(sv[mv]) / np.sum(cv[mv])
        cm = a5["mod_msk_cnt"][ki, :, vi]; sm = a5["mod_msk_sum_phys"][ki, :, vi]
        mm = cm > 0; mae_m = np.sum(sm[mm]) / np.sum(cm[mm])
        gap = mae_m - mae_v
        pct = 100 * gap / mae_v
        print(f"  {lbl:>8s}  {mae_v:10.4f}  {mae_m:10.4f}  {gap:+10.4f}  {pct:+7.1f}%")

In [ ]:
# ── Masked vs visible gap line plot ──
fig, axes = plt.subplots(1, NV, figsize=(4.5 * NV, 3.5))
leads_min = GRID * 10 / 60
a5 = AGG5["v27"]

for vi, (ax, v) in enumerate(zip(axes, VARS)):
    mae_vis, mae_msk = [], []
    for ki in range(K):
        cv = a5["mod_vis_cnt"][ki, :, vi]; sv = a5["mod_vis_sum_phys"][ki, :, vi]
        mv = cv > 0; mae_vis.append(np.sum(sv[mv]) / np.sum(cv[mv]))
        cm = a5["mod_msk_cnt"][ki, :, vi]; sm = a5["mod_msk_sum_phys"][ki, :, vi]
        mm = cm > 0; mae_msk.append(np.sum(sm[mm]) / np.sum(cm[mm]))
    ax.plot(leads_min, mae_vis, label="visible", color="#1F5F6B", linewidth=1.5)
    ax.plot(leads_min, mae_msk, label="masked",  color="#D9663D", linewidth=1.5)
    ax.fill_between(leads_min, mae_vis, mae_msk, alpha=0.15, color="#D9663D")
    ax.set_title(f"{v} [{C.UNITS[v]}]", fontsize=10)
    ax.set_xlabel("Lead time (h)")
    if vi == 0: ax.set_ylabel("MAE")
    ax.grid(alpha=0.3)

axes[0].legend(fontsize=9)
fig.suptitle("MAE Transformer at MR=0.5: MAE vs lead time for stations while masked vs while visible\n"
             "pooled over all windows and 155 stations; the mask is redrawn every window", fontsize=12, y=1.04)
plt.tight_layout()
C.save_fig(fig, "summary_mask_gap")
plt.show(); plt.close(fig)

In [ ]:
# ── 3b: Dense vs MAE at MR=0.0 (controlled masking comparison) ──
print("Dense vs MAE at MR=0.0  (Δ = MAE − Dense)")
print("=" * 65)
a_d = AGG0["v31"]; a_m = AGG0["v27"]
for vi, v in enumerate(VARS):
    print(f"\n  {v} [{C.UNITS[v]}]:")
    print(f"  {'Lead':>8s}  {'Dense':>10s}  {'MAE':>10s}  {'Δ':>10s}  {'Δ %':>8s}")
    for ki, lbl in [(1, "+30m"), (6, "+3h"), (12, "+6h")]:
        cnt_d = a_d["mod_all_cnt"][ki, :, vi]; s_d = a_d["mod_all_sum_phys"][ki, :, vi]
        md_ = cnt_d > 0; mae_d = np.sum(s_d[md_]) / np.sum(cnt_d[md_])
        cnt_m = a_m["mod_all_cnt"][ki, :, vi]; s_m = a_m["mod_all_sum_phys"][ki, :, vi]
        mm = cnt_m > 0; mae_m = np.sum(s_m[mm]) / np.sum(cnt_m[mm])
        delta = mae_m - mae_d
        pct = 100 * delta / mae_d
        print(f"  {lbl:>8s}  {mae_d:10.4f}  {mae_m:10.4f}  {delta:+10.4f}  {pct:+7.1f}%")

## 4. Effect of spatial attention

Blind vs Dense at MR=0.0. Positive difference = Dense is better (spatial attention helps).

In [ ]:
# ── Spatial-attention gain: Blind − Dense ──
a_b = AGG0["v32-blind"]; a_d = AGG0["v31"]
leads_min = GRID * 10 / 60

fig, axes = plt.subplots(1, NV, figsize=(4.5 * NV, 3.5))
for vi, (ax, v) in enumerate(zip(axes, VARS)):
    gain = []
    for ki in range(K):
        cnt_b = a_b["mod_all_cnt"][ki, :, vi]; s_b = a_b["mod_all_sum_phys"][ki, :, vi]
        mb = cnt_b > 0; mae_b = np.sum(s_b[mb]) / np.sum(cnt_b[mb])
        cnt_d = a_d["mod_all_cnt"][ki, :, vi]; s_d = a_d["mod_all_sum_phys"][ki, :, vi]
        md_ = cnt_d > 0; mae_d = np.sum(s_d[md_]) / np.sum(cnt_d[md_])
        gain.append(mae_b - mae_d)
    ax.bar(leads_min, gain, width=0.35, color="#D9663D", alpha=0.8)
    ax.axhline(0, color="k", linewidth=0.5)
    ax.set_title(f"{v} [{C.UNITS[v]}]", fontsize=10)
    ax.set_xlabel("Lead time (h)")
    if vi == 0: ax.set_ylabel("MAE(Blind) − MAE(Dense)")
    ax.grid(alpha=0.3, axis="y")

fig.suptitle("Spatial-attention gain: MAE(Spatially Blind) − MAE(Dense) vs lead time, MR=0, all stations\n"
             "positive = cross-station attention lowers the error", fontsize=12, y=1.04)
plt.tight_layout()
C.save_fig(fig, "summary_spatial_attn_gain")
plt.show(); plt.close(fig)

# Table
print("Spatial-attention gain at key leads (Blind − Dense):")
print("=" * 70)
for vi, v in enumerate(VARS):
    print(f"\n  {v} [{C.UNITS[v]}]:")
    print(f"  {'Lead':>8s}  {'Blind':>10s}  {'Dense':>10s}  {'Δ':>10s}  {'Δ %':>8s}")
    for ki, lbl in [(1, "+30m"), (6, "+3h"), (12, "+6h")]:
        cnt_b = a_b["mod_all_cnt"][ki, :, vi]; s_b = a_b["mod_all_sum_phys"][ki, :, vi]
        mb = cnt_b > 0; mae_b = np.sum(s_b[mb]) / np.sum(cnt_b[mb])
        cnt_d = a_d["mod_all_cnt"][ki, :, vi]; s_d = a_d["mod_all_sum_phys"][ki, :, vi]
        md_ = cnt_d > 0; mae_d = np.sum(s_d[md_]) / np.sum(cnt_d[md_])
        delta = mae_b - mae_d
        pct = 100 * delta / mae_d
        print(f"  {lbl:>8s}  {mae_b:10.4f}  {mae_d:10.4f}  {delta:+10.4f}  {pct:+7.1f}%")

## 5. σₑ comparison: MAE (Huber) vs Probabilistic MAE (NLL)

σₑ = std(ŷ − y), computed per station then reported as pooled and median.

In [ ]:
# ── σₑ per-station, then pooled and median ──
KEY_LEADS_5 = [(1, "+30m"), (6, "+3h"), (12, "+6h")]

def station_sd(a, ki, vi):
    cnt = a["mod_all_cnt"][ki, :, vi]
    sgn = a["mod_all_signed_phys"][ki, :, vi]
    sq  = a["mod_all_sumsq_phys"][ki, :, vi]
    n = np.maximum(cnt, 1)
    sd = np.sqrt(np.maximum(sq / n - (sgn / n)**2, 0))
    return np.where(cnt > 0, sd, np.nan)

print("Per-station σₑ — MEDIAN across stations")
print("=" * 65)
for vi, v in enumerate(VARS):
    print(f"\n  {v} [{C.UNITS[v]}]:")
    print(f"  {'Model':25s}" + "".join(f"  {lbl:>10s}" for _, lbl in KEY_LEADS_5))
    for r in ["v27", "v30-nll"]:
        label = C.MODELS[r][0]
        vals = [f"{np.nanmedian(station_sd(AGG0[r], ki, vi)):10.4f}" for ki, _ in KEY_LEADS_5]
        print(f"  {label:25s}" + "".join(f"  {v}" for v in vals))

# Plot
fig, axes = plt.subplots(1, NV, figsize=(4.5 * NV, 3.5))
leads_min = GRID * 10 / 60
for vi, (ax, v) in enumerate(zip(axes, VARS)):
    for r, ls in [("v27", "-"), ("v30-nll", "--")]:
        label = C.MODELS[r][0]; col = C.MODELS[r][1]
        sd_med = [np.nanmedian(station_sd(AGG0[r], ki, vi)) for ki in range(K)]
        ax.plot(leads_min, sd_med, color=col, linestyle=ls, label=label, linewidth=1.5)
    ax.set_title(f"{v} [{C.UNITS[v]}]", fontsize=10)
    ax.set_xlabel("Lead time (h)")
    if vi == 0: ax.set_ylabel("median σₑ")
    ax.grid(alpha=0.3)

axes[-1].legend(fontsize=9)
fig.suptitle("Error spread vs lead time: median over stations of the per-station error SD σₑ = std(ŷ−y)\n"
             "MAE Transformer (Huber loss) vs Probabilistic MAE (Gaussian NLL), MR=0", fontsize=12, y=1.04)
plt.tight_layout()
C.save_fig(fig, "summary_sd_huber_vs_nll")
plt.show(); plt.close(fig)

## 6. Nearest-neighbour distance and topographic effects

Pearson correlation between station descriptors and per-station MAE at +3h.

In [ ]:
# ── NN distances ──
coords = stn[["easting", "northing"]].values
D = distance_matrix(coords, coords)
np.fill_diagonal(D, np.inf)
nn1 = np.sort(D, axis=1)[:, 0]
nn3 = np.sort(D, axis=1)[:, 2]

print(f"1st NN distance: median = {np.median(nn1)/1000:.1f} km, "
      f"range = [{nn1.min()/1000:.1f}, {nn1.max()/1000:.1f}] km")
print(f"3rd NN distance: median = {np.median(nn3)/1000:.1f} km, "
      f"range = [{nn3.min()/1000:.1f}, {nn3.max()/1000:.1f}] km")

# ── Correlation table ──
descriptors = {
    "3rd-NN dist (km)": nn3 / 1000,
    "height (m)":       stn["height"].values,
    "DEM (m)":          stn["dem"].values,
    "slope":            stn["slope"].values,
    "rel. height":      stn["rel_height"].values,
    "relief 2 km":      stn["relief_2km"].values,
    "TPI n=10":         stn["tpi_n10"].values,
}

ki = REF_KI
print(f"\nPearson r(descriptor, station MAE) at +3h, MR=0.0:")
print("=" * 90)
for r in ["v31", "v27", "v32-blind"]:
    label = C.MODELS[r][0]
    print(f"\n  {label}:")
    header = f"  {'descriptor':20s}" + "".join(f"  {v:>12s}" for v in VARS)
    print(header)
    for dname, dvals in descriptors.items():
        row = f"  {dname:20s}"
        for vi in range(NV):
            cnt = AGG0[r]["mod_all_cnt"][ki, :, vi]
            s   = AGG0[r]["mod_all_sum_phys"][ki, :, vi]
            mae = np.where(cnt > 0, s / np.maximum(cnt, 1), np.nan)
            valid = ~np.isnan(mae) & ~np.isnan(dvals)
            if np.sum(valid) > 10:
                corr = np.corrcoef(dvals[valid], mae[valid])[0, 1]
                row += f"  {corr:+12.3f}"
            else:
                row += f"  {'n/a':>12s}"
        print(row)

In [ ]:
# ── Scatter: 3rd-NN dist vs MAE at +3h, temperature ──
fig, axes = plt.subplots(1, len(ORDER), figsize=(4.0 * len(ORDER), 3.5), sharey=True)
vi = VARS.index("temperature")
ki = REF_KI

for ri, (ax, r) in enumerate(zip(axes, ORDER)):
    a = AGG0[r]; label = LABELS[ri]; col = COLORS[ri]
    cnt = a["mod_all_cnt"][ki, :, vi]
    s   = a["mod_all_sum_phys"][ki, :, vi]
    mae = np.where(cnt > 0, s / np.maximum(cnt, 1), np.nan)
    valid = ~np.isnan(mae)
    ax.scatter(nn3[valid] / 1000, mae[valid], s=15, alpha=0.6, color=col)
    corr = np.corrcoef(nn3[valid], mae[valid])[0, 1]
    ax.set_title(f"{label}\nr = {corr:+.3f}", fontsize=10)
    ax.set_xlabel("3rd-NN dist (km)")
    if ri == 0: ax.set_ylabel(f"MAE temperature [{C.UNITS['temperature']}]")
    ax.grid(alpha=0.3)

fig.suptitle("Per-station temperature MAE at +3 h (MR=0) vs distance to the 3rd-nearest station, one panel per model", fontsize=12, y=1.06)
plt.tight_layout()
C.save_fig(fig, "summary_nn3_vs_mae_temp")
plt.show(); plt.close(fig)

## 7. MAE by terrain class at +3h

In [ ]:
tc_idx = C.terrain_class_indices(stn)
tc_names = list(tc_idx.keys())
ki = REF_KI

print(f"Terrain classes: {', '.join(f'{k} ({len(v)})' for k, v in tc_idx.items())}")
print()

for vi, v in enumerate(VARS):
    print(f"{v} [{C.UNITS[v]}]:")
    header = f"  {'Model':25s}" + "".join(f"  {tc:>22s}" for tc in tc_names)
    print(header)
    for ri, r in enumerate(ORDER):
        a = AGG0[r]; label = LABELS[ri]
        vals = []
        for tc in tc_names:
            idx = tc_idx[tc]
            cnt = a["mod_all_cnt"][ki, idx, vi]
            s   = a["mod_all_sum_phys"][ki, idx, vi]
            m = cnt > 0
            mae = np.sum(s[m]) / np.sum(cnt[m]) if np.any(m) else np.nan
            vals.append(mae)
        print(f"  {label:25s}" + "".join(f"  {v:22.4f}" for v in vals))
    print()

In [ ]:
# ── Grouped bar: terrain class × model for temperature and pressure ──
tc_idx = C.terrain_class_indices(stn)
tc_names = list(tc_idx.keys())
ki = REF_KI
plot_vars = ["temperature", "pressure", "humidity"]

fig, axes = plt.subplots(1, len(plot_vars), figsize=(6 * len(plot_vars), 4))
x = np.arange(len(tc_names))
w = 0.15

for pi, (ax, v) in enumerate(zip(axes, plot_vars)):
    vi = VARS.index(v)
    for ri, r in enumerate(ORDER):
        a = AGG0[r]
        vals = []
        for tc in tc_names:
            idx = tc_idx[tc]
            cnt = a["mod_all_cnt"][ki, idx, vi]
            s   = a["mod_all_sum_phys"][ki, idx, vi]
            m = cnt > 0
            vals.append(np.sum(s[m]) / np.sum(cnt[m]))
        ax.bar(x + ri * w, vals, w, color=COLORS[ri], label=LABELS[ri], alpha=0.85)
    ax.set_xticks(x + 2 * w)
    ax.set_xticklabels(tc_names, fontsize=8, rotation=15, ha="right")
    ax.set_ylabel(f"MAE [{C.UNITS[v]}]")
    ax.set_title(f"{v}", fontsize=11)
    ax.grid(alpha=0.3, axis="y")

axes[-1].legend(fontsize=7, loc="upper left")
fig.suptitle(f"MAE at +3h by terrain class (MR=0.0)", fontsize=13, y=1.02)
plt.tight_layout()
C.save_fig(fig, "summary_terrain_class")
plt.show(); plt.close(fig)

## Key observations

1. **Ranking at MR=0.0 (Δ>0):** Dense > MAE ≈ Prob. MAE > Blind > LSTM, consistently across variables and leads.

2. **Lead-time growth:** errors grow roughly linearly; relative model ranking stays stable.

3. **Masking degradation:** the masked–visible gap at Δ=0 is enormous (>1000% for temperature) but collapses to <2% by +6h — masking primarily tests spatial reconstruction.

4. **Mask-training cost at MR=0.0:** MAE is 5–15% worse than Dense when all stations are present — the price of optimising for a harder task.

5. **Spatial attention:** Dense beats Blind by 10–25% at +3h across all variables, with the gain growing with lead time.

6. **NLL vs Huber σₑ:** Prob. MAE has *higher* σₑ at most leads — the NLL objective allows larger errors where uncertainty is high, trading point-prediction stability for calibrated confidence intervals.

7. **NN distance:** weak positive correlation with MAE for temperature (r≈0.2 in spatially aware models, ~0 in Blind) — isolated stations do benefit from spatial attention, but the effect is modest.

8. **Topography:** pressure MAE is strongly anti-correlated with height (r≈−0.9); terrain class matters more for humidity (ridges worst) than temperature (valleys worst).